# Autograd: let PyTorch do the backward pass

The dumpling shop's tiny network promised a 23-minute wait last Friday (30 orders, raining). The real wait was
35 minutes. Last lesson you found the gradients by hand. Here you let PyTorch find them with one
`loss.backward()`, check them against your hand numbers, and break and fix a two-day check.
Same weights, same days and same code as the lesson.

About 5 minutes. No account, key or GPU needed. Run the cells from top to bottom.

## Step 1: the network's numbers

This cell imports PyTorch and writes down the shop's data: Friday and Tuesday (orders in tens, rain 1 or 0,
real wait in minutes), and the gradients of `w1` you found by hand. `params()` makes fresh weight tensors with
`requires_grad=True`. Nothing here is random, but we seed PyTorch anyway.

**Look for:** the PyTorch version. The lesson was run with 2.15.0; any recent version prints the same numbers.

In [1]:
import warnings
import torch

torch.manual_seed(0)
print("torch", torch.__version__)

FRIDAY_ORDERS_RAIN, FRIDAY_WAIT = [3.0, 1.0], 35.0   # 30 orders, raining; real wait 35 minutes
days = {"friday": ([3.0, 1.0], 35.0), "tuesday": ([1.0, 0.0], 14.0)}
by_hand = {"friday": [[-72.0, -24.0], [-36.0, -12.0]], "tuesday": [[-12.0, 0.0], [0.0, 0.0]]}

def params():
    return (torch.tensor([[2.0, 1.0], [1.0, 4.0]], requires_grad=True), torch.tensor([1.0, -2.0], requires_grad=True),
            torch.tensor([2.0, 1.0], requires_grad=True), torch.tensor(2.0, requires_grad=True))

def friday_loss(w1, b1, v, c):
    orders_rain = torch.tensor([3.0, 1.0])
    pred = v @ torch.relu(w1 @ orders_rain + b1) + c
    return 0.5 * (pred - 35.0) ** 2

print("hand gradients of w1:", by_hand)

torch 2.15.0
hand gradients of w1: {'friday': [[-72.0, -24.0], [-36.0, -12.0]], 'tuesday': [[-12.0, 0.0], [0.0, 0.0]]}


## Step 2: the forward pass

Now the forward pass for Friday, written out line by line. Each result made from a tensor with
`requires_grad=True` gets a `grad_fn`: a note of the operation that made it.

**Look for:** `hidden` = [8.0, 5.0], `pred` = 23.0 and `loss` = 72.0, the same as your hand forward pass. Each has
a `grad_fn` (`ReluBackward0`, `AddBackward0`, `MulBackward0`). `w1.grad` is still None.

In [2]:
w1 = torch.tensor([[2.0, 1.0], [1.0, 4.0]], requires_grad=True)
b1 = torch.tensor([1.0, -2.0], requires_grad=True)
v = torch.tensor([2.0, 1.0], requires_grad=True)
c = torch.tensor(2.0, requires_grad=True)
orders_rain = torch.tensor(FRIDAY_ORDERS_RAIN)
z = w1 @ orders_rain + b1
hidden = torch.relu(z)
pred = v @ hidden + c
loss = 0.5 * (pred - FRIDAY_WAIT) ** 2

print("hidden", hidden.tolist(), "grad_fn", type(hidden.grad_fn).__name__)
print("pred  ", pred.item(), "grad_fn", type(pred.grad_fn).__name__)
print("loss  ", loss.item(), "grad_fn", type(loss.grad_fn).__name__)
print("w1.grad before backward:", w1.grad)

hidden [8.0, 5.0] grad_fn ReluBackward0
pred   23.0 grad_fn AddBackward0
loss   72.0 grad_fn MulBackward0
w1.grad before backward: None


## Step 3: one call to `backward()`

`loss.backward()` walks the recorded notes from the loss back to the weights and fills each weight's `.grad`.
You write no backward code.

**Look for:** `w1.grad` = [[-72.0, -24.0], [-36.0, -12.0]], `v.grad` = [-96.0, -60.0], `c.grad` = -12.0, and
`matches the hand numbers: True`.

In [3]:
loss.backward()
print("w1.grad", w1.grad.tolist())
print("b1.grad", b1.grad.tolist())
print("v.grad ", v.grad.tolist())
print("c.grad ", c.grad.item())
print("matches the hand numbers:", w1.grad.tolist() == by_hand["friday"])

w1.grad [[-72.0, -24.0], [-36.0, -12.0]]
b1.grad [-24.0, -12.0]
v.grad  [-96.0, -60.0]
c.grad  -12.0
matches the hand numbers: True


## Step 4: who gets a `.grad`?

Nine tensors took part. This cell prints each one's `requires_grad`, `is_leaf`, `grad_fn` and `.grad`.
Reading `.grad` on an in-between tensor makes PyTorch print a warning; we catch it and print it once.

**Look for:** only the leaves you marked (`w1`, `b1`, `v`, `c`) have a `.grad`. `orders_rain` is a leaf but does not
require grad. `z`, `hidden`, `pred` and `loss` have a `grad_fn` and `.grad` None, and the warning says
"is not a leaf Tensor".

In [4]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    rows = []
    for name, t in [("orders_rain", orders_rain), ("w1", w1), ("b1", b1), ("v", v), ("c", c),
                    ("z", z), ("hidden", hidden), ("pred", pred), ("loss", loss)]:
        rows.append((name, t.requires_grad, t.is_leaf,
                     None if t.grad_fn is None else type(t.grad_fn).__name__,
                     None if t.grad is None else t.grad.tolist()))

print(f"{'tensor':<12}{'requires_grad':<15}{'is_leaf':<9}{'grad_fn':<15}grad")
for name, rg, leaf, fn, g in rows:
    print(f"{name:<12}{str(rg):<15}{str(leaf):<9}{str(fn):<15}{g}")
if caught:
    message = str(caught[0].message).split(" (Triggered internally")[0]   # drop the build path at the end
    print()
    print(f"WARNING ({len(caught)} times):", message)

tensor      requires_grad  is_leaf  grad_fn        grad
orders_rain False          True     None           None
w1          True           True     None           [[-72.0, -24.0], [-36.0, -12.0]]
b1          True           True     None           [-24.0, -12.0]
v           True           True     None           [-96.0, -60.0]
c           True           True     None           -12.0
z           True           False    AddBackward0   None
hidden      True           False    ReluBackward0  None
pred        True           False    AddBackward0   None
loss        True           False    MulBackward0   None

WARNING (4 times): The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pyt

## Step 5: the recorded graph

Every `grad_fn` links to the notes it came from (`next_functions`). This cell starts at `loss.grad_fn` and prints
the whole record, two spaces per level.

**Look for:** `MulBackward0` (the ½·) at the top, then `PowBackward0` (the square), `SubBackward0` (the − 35),
`AddBackward0` (+ c), `DotBackward0` (`v @ hidden`), `ReluBackward0`, `AddBackward0` (+ b1) and `MvBackward0`
(`w1 @ orders_rain`). Each `AccumulateGrad` is a weight: the place where a gradient is written into its `.grad`.
There are four of them.

In [5]:
def walk(fn, depth):
    if fn is None:
        return
    print("  " * depth + type(fn).__name__)
    for nxt, _ in fn.next_functions:
        walk(nxt, depth + 1)

walk(loss.grad_fn, 0)

MulBackward0
  PowBackward0
    SubBackward0
      AddBackward0
        DotBackward0
          AccumulateGrad
          ReluBackward0
            AddBackward0
              MvBackward0
                AccumulateGrad
              AccumulateGrad
        AccumulateGrad


## Step 6: keep one in-between gradient

`hidden` is not a leaf, so its gradient is used on the way back and dropped. `hidden.retain_grad()` before
`backward()` asks PyTorch to keep it.

**Look for:** `hidden.grad` = [-24.0, -12.0], the slope at `hidden` from your hand walk.

In [6]:
w1, b1, v, c = params()
hidden = torch.relu(w1 @ torch.tensor(FRIDAY_ORDERS_RAIN) + b1)
hidden.retain_grad()
loss = 0.5 * (v @ hidden + c - FRIDAY_WAIT) ** 2
loss.backward()
print("hidden.grad", hidden.grad.tolist())

hidden.grad [-24.0, -12.0]


## Step 7: `backward()` twice on the same loss

This cell calls `loss.backward()` a second time on the same `loss`. It is meant to fail, so the call is inside
`try/except` and the error is printed. Then, for comparison, the same with `retain_graph=True` on the first call.

**Look for:** "RuntimeError: Trying to backward through the graph a second time". `w1.grad` is unchanged after the
error. With `retain_graph=True` the second call works and `w1.grad` doubles to [[-144.0, -48.0], [-72.0, -24.0]].

In [7]:
w1, b1, v, c = params()
loss = friday_loss(w1, b1, v, c)
loss.backward()
print("after 1st backward  w1.grad =", w1.grad.tolist(), " c.grad =", c.grad.item())
try:
    loss.backward()
except RuntimeError as e:
    print("RuntimeError:", e)
print("after the error     w1.grad =", w1.grad.tolist(), " c.grad =", c.grad.item())

w1, b1, v, c = params()
loss = friday_loss(w1, b1, v, c)
loss.backward(retain_graph=True)
loss.backward()
print("retain_graph=True, after 2 backwards  w1.grad =", w1.grad.tolist(), " c.grad =", c.grad.item())

after 1st backward  w1.grad = [[-72.0, -24.0], [-36.0, -12.0]]  c.grad = -12.0
RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.
after the error     w1.grad = [[-72.0, -24.0], [-36.0, -12.0]]  c.grad = -12.0
retain_graph=True, after 2 backwards  w1.grad = [[-144.0, -48.0], [-72.0, -24.0]]  c.grad = -24.0


## Step 8: `backward()` adds, it does not replace

A new forward pass builds a new graph, so a new `backward()` works. But it **adds** into `.grad`. First three runs
without clearing, then two runs that set every `.grad` to None first.

**Look for:** without clearing, `w1.grad[0][0]` goes -72.0, -144.0, -216.0 and `c.grad` -12.0, -24.0, -36.0. With
clearing, both runs print the same [[-72.0, -24.0], [-36.0, -12.0]].

In [8]:
print("no clearing")
w1, b1, v, c = params()
for run in (1, 2, 3):
    loss = friday_loss(w1, b1, v, c)
    loss.backward()
    print(f"after run {run}  w1.grad =", w1.grad.tolist(), " c.grad =", c.grad.item())

print("clear the receipts first: w1.grad = None etc.")
w1, b1, v, c = params()
for run in (1, 2):
    for p in (w1, b1, v, c):
        p.grad = None
    loss = friday_loss(w1, b1, v, c)
    loss.backward()
    print(f"after run {run}  w1.grad =", w1.grad.tolist(), " c.grad =", c.grad.item())

no clearing
after run 1  w1.grad = [[-72.0, -24.0], [-36.0, -12.0]]  c.grad = -12.0
after run 2  w1.grad = [[-144.0, -48.0], [-72.0, -24.0]]  c.grad = -24.0
after run 3  w1.grad = [[-216.0, -72.0], [-108.0, -36.0]]  c.grad = -36.0
clear the receipts first: w1.grad = None etc.
after run 1  w1.grad = [[-72.0, -24.0], [-36.0, -12.0]]  c.grad = -12.0
after run 2  w1.grad = [[-72.0, -24.0], [-36.0, -12.0]]  c.grad = -12.0


## Step 9: broken on purpose, Tuesday fails

A teammate checks two days with the same four weight tensors and never clears `.grad`. Tuesday was 10 orders,
dry, real wait 14 minutes.

**Look for:** Friday says OK. Tuesday says MISMATCH with [[-84.0, -24.0], [-36.0, -12.0]]: -84 = -72 (Friday) + -12
(Tuesday), and the other three cells are Friday's leftovers. Nothing crashed.

In [9]:
w1, b1, v, c = params()

def check(day):
    orders_rain, wait = days[day]
    pred = v @ torch.relu(w1 @ torch.tensor(orders_rain) + b1) + c
    loss = 0.5 * (pred - wait) ** 2
    loss.backward()
    match = w1.grad.tolist() == by_hand[day]
    print(day, "w1.grad", w1.grad.tolist(), "by hand", by_hand[day], "OK" if match else "MISMATCH")

for day in ["friday", "tuesday"]:
    check(day)

friday w1.grad [[-72.0, -24.0], [-36.0, -12.0]] by hand [[-72.0, -24.0], [-36.0, -12.0]] OK
tuesday w1.grad [[-84.0, -24.0], [-36.0, -12.0]] by hand [[-12.0, 0.0], [0.0, 0.0]] MISMATCH


## Step 10: the fix

`check_fixed` sets every weight's `.grad` to None before each check, so each day starts a fresh sum. In a training
loop, an optimizer's `zero_grad()` does this for you.

**Look for:** both days say OK. Tuesday prints `-0.0` in one cell; that equals 0, so the check still passes.

In [10]:
w1, b1, v, c = params()

def check_fixed(day):
    for p in (w1, b1, v, c):
        p.grad = None
    check(day)

for day in ["friday", "tuesday"]:
    check_fixed(day)

friday w1.grad [[-72.0, -24.0], [-36.0, -12.0]] by hand [[-72.0, -24.0], [-36.0, -12.0]] OK
tuesday w1.grad [[-12.0, -0.0], [0.0, 0.0]] by hand [[-12.0, 0.0], [0.0, 0.0]] OK


## Step 11: where autograd cannot help

Four short tries on Friday's network: no `requires_grad` anywhere, rounding the estimate before the loss, a NumPy
step in the middle, and serving an estimate under `torch.no_grad()`. The two that fail are inside `try/except`.

**Look for:** 1 prints "does not require grad". 2 prints loss 72.0 but every gradient 0.0: rounding is flat, so its
slope is 0. 3 prints "Can't call numpy() on Tensor that requires grad". 4 prints pred 23.0 with `requires_grad`
False and `grad_fn` None: nothing was recorded.

In [11]:
orders_rain = torch.tensor([3.0, 1.0])

print("1: forgot requires_grad on every weight")
w1, b1, v, c = (t.detach() for t in params())
loss = 0.5 * (v @ torch.relu(w1 @ orders_rain + b1) + c - 35.0) ** 2
try:
    loss.backward()
except RuntimeError as e:
    print("   RuntimeError:", e)

print("2: the estimate rounded to whole minutes before the loss")
w1, b1, v, c = params()
pred = torch.round(v @ torch.relu(w1 @ orders_rain + b1) + c)
loss = 0.5 * (pred - 35.0) ** 2
loss.backward()
print("   loss", loss.item(), "w1.grad", w1.grad.tolist(), "c.grad", c.grad.item())

print("3: one step done in NumPy in the middle")
w1, b1, v, c = params()
try:
    hidden = torch.relu(w1 @ orders_rain + b1).numpy()
except RuntimeError as e:
    print("   RuntimeError:", e)

print("4: serving an estimate under torch.no_grad()")
w1, b1, v, c = params()
with torch.no_grad():
    pred = v @ torch.relu(w1 @ orders_rain + b1) + c
print("   pred", pred.item(), "requires_grad", pred.requires_grad, "grad_fn", pred.grad_fn)

1: forgot requires_grad on every weight
   RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn
2: the estimate rounded to whole minutes before the loss
   loss 72.0 w1.grad [[0.0, 0.0], [0.0, 0.0]] c.grad 0.0
3: one step done in NumPy in the middle
   RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.
4: serving an estimate under torch.no_grad()
   pred 23.0 requires_grad False grad_fn None


## Now change it

Three small experiments. Each cell has one line marked `# <- edit`. Change it, run the cell, and compare.

### Experiment 1: a perfect Friday

Change the real wait on the marked line from `35.0` to `23.0`, the network's own estimate.

**What should change:** `loss` becomes 0.0 and every gradient becomes 0.0. The blame is pred − actual times
local slopes; with no miss there is nothing to blame.

In [12]:
wait = 35.0   # <- edit: try 23.0
w1, b1, v, c = params()
pred = v @ torch.relu(w1 @ torch.tensor([3.0, 1.0]) + b1) + c
loss = 0.5 * (pred - wait) ** 2
loss.backward()
print("loss", loss.item())
print("w1.grad", w1.grad.tolist(), " v.grad", v.grad.tolist(), " c.grad", c.grad.item())

loss 72.0
w1.grad [[-72.0, -24.0], [-36.0, -12.0]]  v.grad [-96.0, -60.0]  c.grad -12.0


### Experiment 2: check Tuesday first

Swap the order on the marked line to `["tuesday", "friday"]`. `.grad` is still never cleared.

**What should change:** now Tuesday says OK and Friday says MISMATCH with [[-84.0, -24.0], [-36.0, -12.0]]. Changing
the order does not fix anything; the leftover just moves to the second day.

In [13]:
w1, b1, v, c = params()
for day in ["friday", "tuesday"]:   # <- edit: try ["tuesday", "friday"]
    check(day)

friday w1.grad [[-72.0, -24.0], [-36.0, -12.0]] by hand [[-72.0, -24.0], [-36.0, -12.0]] OK
tuesday w1.grad [[-84.0, -24.0], [-36.0, -12.0]] by hand [[-12.0, 0.0], [0.0, 0.0]] MISMATCH


### Experiment 3: rain on Tuesday

Change Tuesday's rain on the marked line from `0.0` to `1.0`. The cell clears `.grad` first, so the numbers are
Tuesday's alone.

**What should change:** with rain, z2 = 1·1 + 4·1 − 2 = 3 is above zero, so h2's ReLU is on and the second row of
`w1.grad` is no longer zero: you should see [[-2.0, -2.0], [-1.0, -1.0]]. On a dry Tuesday h2 is off and gets
no blame.

In [14]:
tuesday = [1.0, 0.0]   # <- edit: try [1.0, 1.0]
w1, b1, v, c = params()
pred = v @ torch.relu(w1 @ torch.tensor(tuesday) + b1) + c
loss = 0.5 * (pred - 14.0) ** 2
loss.backward()
print("pred", pred.item(), " w1.grad", w1.grad.tolist())

pred 8.0  w1.grad [[-12.0, -0.0], [0.0, 0.0]]


## What you should see

- `w1.grad` prints None before `loss.backward()`.
- After one backward, `w1.grad` is [[-72.0, -24.0], [-36.0, -12.0]] and `c.grad` is -12.0: your hand numbers.
- `hidden.grad` is None, with a warning about non-leaf tensors, until you call `hidden.retain_grad()`; then it is [-24.0, -12.0].
- A second `loss.backward()` on the same loss raises "Trying to backward through the graph a second time".
- Forward and backward again without clearing: `w1.grad[0][0]` goes -72.0, -144.0, -216.0.
- With `.grad` set to None before each day, Tuesday's `w1.grad` matches the hand numbers (it may print `-0.0` in one cell; that equals 0).